In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, floor, months_between, get
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.dim_customer")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "dim_customer"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_customer.ipynb"
_silver_table = "silver.customer"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.dim_customer


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver schema

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 5
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------------+---------------------+--------------------------+---------------------+
|customer_id|store |channel|first_name|middle_name|last_name|phone_number|email             |gender|date_of_birth|effective_start_date|is_current|is_deleted|source_file     |created_on                |created_by           |modified_on               |modified_by          |
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------------+---------------------+--------------------------+---------------------+
|CUST001    |AMZ_US|MKT    |John      |null       |Doe      |9991112222  |john@gmail.com    |Male  |1988-05-10   |2024-01-01          |false  

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ - [ERROR] : gold.dim_customer | Record cannot be deleted and current | Failed count : 1
SPARK-APP: DQ - [WARN] : gold.dim_customer | Invalid gender value | Failed count : 5
SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 4


In [12]:
# Generating surrogate key and business key

df_sk = df.withColumn("customer_sk", xxhash64(concat_ws("||", "customer_id", "store", "channel", "effective_start_date")).cast("bigint")) \
          .withColumn("customer_bk", xxhash64(concat_ws("||", "customer_id", "store", "channel")).cast("bigint"))

df_sk.show(truncate = False)

+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------------+---------------------+--------------------------+---------------------+--------------------+--------------------+
|customer_id|store |channel|first_name|middle_name|last_name|phone_number|email             |gender|date_of_birth|effective_start_date|is_current|is_deleted|source_file     |created_on                |created_by           |modified_on               |modified_by          |customer_sk         |customer_bk         |
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------------+---------------------+--------------------------+---------------------+--------------------+--------------------+
|CUST001    |AMZ_US|MKT    |John      |null       |Doe 

In [13]:
#Derive additional columns

df_sk = df_sk.withColumn("age", floor(months_between(current_date(), col("date_of_birth")) / 12))

df_sk.show()

+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+
|customer_id| store|channel|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|effective_start_date|is_current|is_deleted|     source_file|          created_on|          created_by|         modified_on|         modified_by|         customer_sk|         customer_bk|age|
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|    john@gm

In [14]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_channel_store = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")

df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_channel_store, how="inner", on=df_sk.channel==df_channel_store.channel_code) \
                 .select("customer_sk", "customer_bk", "customer_id", "store_sk", "channel_sk", "first_name", "middle_name", 
                        "last_name", "phone_number", "email", "gender", "date_of_birth", "age", "effective_start_date", "is_current", "is_deleted", "source_file")

df_silver.show(truncate = False)

+--------------------+--------------------+-----------+--------+----------+----------+-----------+---------+------------+------------------+------+-------------+---+--------------------+----------+----------+----------------+
|customer_sk         |customer_bk         |customer_id|store_sk|channel_sk|first_name|middle_name|last_name|phone_number|email             |gender|date_of_birth|age|effective_start_date|is_current|is_deleted|source_file     |
+--------------------+--------------------+-----------+--------+----------+----------+-----------+---------+------------+------------------+------+-------------+---+--------------------+----------+----------+----------------+
|1649233548653759539 |-7604987655276774799|CUST001    |2       |1         |John      |null       |Doe      |9991112222  |john@gmail.com    |Male  |1988-05-10   |37 |2024-01-01          |false     |false     |dim_customer.csv|
|-798410774282725028 |-7604987655276774799|CUST001    |2       |1         |John      |null      

In [15]:
#Truncate table for full load
dt_dim_customer = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_dim_customer.delete()
    dt_dim_customer.vacuum(0)



In [16]:
#SCD2 - Merge 1 for dates already in dim table
#Assuming file can have mixed cases for sending records - could be with history versions or sometimes only latest rows

dt_dim_customer.alias("t").merge(
    df_silver.alias("s"), 
    "t.customer_bk = s.customer_bk and t.is_current = true and s.effective_start_date > t.effective_start_date"
    ).whenMatchedUpdate(set={
    "effective_end_date" : "date_sub(s.effective_start_date, 1)",
    "is_current" : "false",
    "modified_on" : "current_timestamp()",
    "modified_by" : f"'{_script_name}'"
}).execute()

print("SPARK-APP: Merge1 completed")

SPARK-APP: Merge1 completed


In [17]:
dt_dim_customer.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows").show(1, False)

+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|0      |null           |null                 |null                |null         |
+-------+---------------+---------------------+--------------------+-------------+



In [18]:

window = Window.partitionBy("customer_bk").orderBy("effective_start_date")

df_silver_nextDate = df_silver.withColumn("next_date", lead("effective_start_date").over(window)) \
                              .withColumn("effective_end_date", when(col("next_date").isNotNull(), date_sub("next_date",1)).otherwise(lit(None).cast("date"))) \
                              .withColumn("is_current", col("next_date").isNull()) \
                              .drop("next_date")

In [19]:
df_silver_nextDate.show()
df_silver_nextDate.printSchema()

+--------------------+--------------------+-----------+--------+----------+----------+-----------+---------+------------+------------------+------+-------------+---+--------------------+----------+----------+----------------+------------------+
|         customer_sk|         customer_bk|customer_id|store_sk|channel_sk|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|age|effective_start_date|is_current|is_deleted|     source_file|effective_end_date|
+--------------------+--------------------+-----------+--------+----------+----------+-----------+---------+------------+------------------+------+-------------+---+--------------------+----------+----------+----------------+------------------+
| 1649233548653759539|-7604987655276774799|    CUST001|       2|         1|      John|       null|      Doe|  9991112222|    john@gmail.com|  Male|   1988-05-10| 37|          2024-01-01|     false|     false|dim_customer.csv|        2024-06-30|
| -79841077428272502

In [20]:
# Merge 2 for new dates not in dim table
dt_dim_customer.alias("t").merge(
    df_silver_nextDate.alias("s"),
    "t.customer_bk = s.customer_bk and s.effective_start_date = t.effective_start_date"
).whenNotMatchedInsert(values = {
    "customer_sk": "s.customer_sk", 
    "customer_bk": "s.customer_bk",
    "customer_id": "s.customer_id",
    "store_sk"   : "s.store_sk",
    "channel_sk" : "s.channel_sk",
    "first_name" : "s.first_name",
    "middle_name": "s.middle_name",
    "last_name"  : "s.last_name",
    "phone_number" : "s.phone_number",
    "email"      : "s.email",
    "gender"     : "s.gender",
    "date_of_birth" : "s.date_of_birth",
    "age"        : "s.age",
    "effective_start_date" : "s.effective_start_date",
    "effective_end_date" : "s.effective_end_date",
    "is_current"  : "s.is_current",
    "is_deleted"  : "s.is_deleted",
    "source_file" : "s.source_file",
    "created_on"  : "current_timestamp()",
    "created_by"  : f"'{_script_name}'",
    "modified_on" : "current_timestamp()",
    "modified_by" : f"'{_script_name}'"
}).execute()
                     
print("SPARK-APP: Merge2 completed")    

SPARK-APP: Merge2 completed


In [21]:
hist = dt_dim_customer.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           4288|                    4|                   0|            4|
+-------+---------------+---------------------+--------------------+-------------+



In [22]:
spark.read.table("gold.dim_customer").show()

+--------------------+--------------------+-----------+--------+----------+----------+-----------+---------+------------+------------------+------+-------------+---+--------------------+------------------+----------+----------+--------------------+-------------------+--------------------+-------------------+----------------+
|         customer_sk|         customer_bk|customer_id|store_sk|channel_sk|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|age|effective_start_date|effective_end_date|is_current|is_deleted|          created_on|         created_by|         modified_on|        modified_by|     source_file|
+--------------------+--------------------+-----------+--------+----------+----------+-----------+---------+------------+------------------+------+-------------+---+--------------------+------------------+----------+----------+--------------------+-------------------+--------------------+-------------------+----------------+
| 16492335486537595

In [23]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [24]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+------------+----------+----------+--------------------+-----------+------------+--------------------+-------------------+--------------------+-------------------+
|layer|schema_name|  table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|         created_by|         modified_on|        modified_by|
+-----+-----------+------------+----------+----------+--------------------+-----------+------------+--------------------+-------------------+--------------------+-------------------+
| gold|       gold|dim_customer|delta-load|     merge|2026-01-29 02:31:...|    success|           4|2026-01-29 02:31:...|gold_customer.ipynb|2026-01-29 02:31:...|gold_customer.ipynb|
+-----+-----------+------------+----------+----------+--------------------+-----------+------------+--------------------+-------------------+--------------------+-------------------+



In [25]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [26]:
spark.stop()